In [2]:
import numpy as np
def get_random_action(*args):
    return np.random.choice([0,1])


In [5]:

no_op = lambda *args, **kwargs: None
def pass_thru(state, *args, **kwargs): return state


# state = init_agent()
init_agent = no_op

# state = reset_agent()
reset_agent = no_op

# action = get_action(state, observation, act_key)
get_action = get_random_action

# state = update(state, observation, reward, terminated, truncated, info)
update = pass_thru

model_name = "Uniform random baseline"

In [6]:
import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np
from tqdm import tqdm

env = gym.make(
    "CartPole-v1",
    render_mode="none",
)

N_TRIALS = 100
N_REPS = 10

# prng key
key = jax.random.PRNGKey(0)

all_trial_results = []

for rep in range(N_REPS):
    # independent key per rep, derived from rep index via fold_in
    key = jax.random.fold_in(jax.random.PRNGKey(0), rep)

    # TODO
    state = init_agent()

    trial_results = []

    pbar = tqdm(range(N_TRIALS), desc=f"repeat {rep}")
    for trial_idx in pbar:
        observation, info = env.reset()

        # TODO optionally reset the agent
        state = reset_agent()

        episode_over = False
        steps = 0

        while not episode_over:
            # this is the nice jax way to get a new random seed which is properly independent
            key, act_key = jax.random.split(key)

            # TODO get action (use act_key if it is random)
            action = get_action(state, observation, act_key)

            # execute action in env
            observation, reward, terminated, truncated, info = env.step(action)
            steps += 1

            # TODO do something with the new info
            state = update(state, observation, reward, terminated, truncated, info)

            episode_over = terminated or truncated

        trial_results.append(steps)
        pbar.set_postfix({"steps": steps})
    
    pbar.close()
    all_trial_results.append(trial_results)

env.close()

all_trial_results = np.array(all_trial_results)  # shape (N_REPS, N_TRIALS)
np.save(f"{model_name}_100_trial_results.npy", all_trial_results)

# Compute average + std steps alive at the last (100th) trial across reps
final_trial_steps = all_trial_results[:, -1]  # shape (N_REPS,)
mean_final = final_trial_steps.mean()
std_final = final_trial_steps.std()
print(f"Final trial (#{N_TRIALS}) steps alive across {N_REPS} reps: "
      f"{mean_final:.2f} ± {std_final:.2f}")

/Users/miranda/miniconda3/envs/nmm_finalproject/lib/python3.12/site-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='none' that is not in the possible render_modes (['human', 'rgb_array']).
  logger.warn(
repeat 0:   0%|          | 0/100 [00:00<?, ?it/s]

repeat 9: 100%|██████████| 100/100 [00:00<00:00, 749.21it/s, steps=11]


Final trial (#100) steps alive across 10 reps: 17.70 ± 4.31


In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_trial_results(model_name, mean_curve, std_curve=None):
    """Plot mean steps-until-failure per trial, with optional std band."""
    trials = np.arange(len(mean_curve))
    fig = go.Figure()
    title="{model_name} Learning Curve"

    if std_curve is not None:
        # Upper bound (invisible line, anchors the fill)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve + std_curve,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))
        # Lower bound (fills down to upper bound)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve - std_curve,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(0, 100, 200, 0.2)",
            name="±1 std",
            hoverinfo="skip",
        ))

    # Mean line on top
    fig.add_trace(go.Scatter(
        x=trials,
        y=mean_curve,
        mode="lines+markers",
        name="Mean steps until failure",
        line=dict(color="rgb(0, 100, 200)"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Trial",
        yaxis_title="Steps until failure",
        template="plotly_white",
    )
    fig.show()

mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)
plot_trial_results(model_name, mean_curve, std_curve)